In [ ]:
import os
import torch
import warnings
import torch.nn as nn
from tqdm import tqdm
from einops import rearrange
import matplotlib.pyplot as plt
import torch.nn.functional as F
from overcomplete.sae import TopKSAE
from domainbed.networks import Identity
from timm.layers import SelectAdaptivePool2d
from domainbed.algorithms import DANN, CORAL, Mixup, MMD, IRM, ERM, SagNet
from torch.optim.lr_scheduler import SequentialLR, LinearLR, CosineAnnealingLR



from lib.loaders import load_backbone
from lib.parser import get_top_k_steps
from lib.utils import extract_features
from lib.data_handlers import  Load_PACS
from lib.gpu_pacs import get_pacs_gpuloader, get_pacs_standard_loader



device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
algo_classes = {"DANN": DANN, "CORAL": CORAL, "Mixup": Mixup, "MMD": MMD, "IRM": IRM, "ERM": ERM, "SagNet": SagNet}
warnings.filterwarnings("ignore", category=UserWarning)
torch.cuda.empty_cache()

In [ ]:
model_dir = r"C:\Users\sproj_ha\Desktop\SGen_Vision_Interp\Vision_Interp\PACS_ResNet_Sketch_Test_Only\ERM_ResNet_T3"
best_nonoracle_steps = get_top_k_steps(os.path.join(model_dir, "out.txt"), envs=[0, 1, 2], k=1)
best_oracle_steps = get_top_k_steps(os.path.join(model_dir, "out.txt"), envs=[0, 1, 2, 3], k=1)


print(f"Best Non-Oracle: {best_nonoracle_steps}")
print(f"Best Oracle: {best_oracle_steps}")

backbones = {}
for step in set(best_nonoracle_steps).union(set(best_oracle_steps)):
    backbones[step] = load_backbone("ERM_ResNet_T3", os.path.join(model_dir, f"model_step{step}.pkl"))

In [ ]:
class Normalizer(nn.Module): 
    def __init__(self, model, dataset, domains=None):
        super().__init__() 

        self.dataset = dataset
        if self.dataset == 'PACS':
            dl, _ = Load_PACS(domains=domains, batch_size=1024)
            x, _ = next(iter(dl))
            
            model.to(device)
            activations = extract_features(model, x.to(device))

        flat = activations.flatten()
        
        self.register_buffer('mean', flat.mean().detach())
        self.register_buffer('std', flat.std().detach())
                
    def forward(self, activations): 
        activations = (activations - self.mean)
        activations = activations / (self.std + 1e-12)
        return activations

    def denormalize(self, normalized_activations):
        return (normalized_activations * (self.std + 1e-12)) + self.mean

In [ ]:
SAEs = torch.load(r"C:\Users\sproj_ha\Desktop\SGen_Vision_Interp\Vision_Interp\SAEs\normalization_testing\USAE_ERM_Multi_test_3300_2100.pt", weights_only=False)
print(SAEs.keys())

## R Score

In [ ]:
import math
import torch
from tqdm import tqdm
from einops import rearrange
from lib.data_handlers import Load_PACS
import json
import os
import json
from collections import defaultdict, Counter
from overcomplete.visualization.plot_utils import (interpolate_cv2, get_image_dimensions, show)
from overcomplete.visualization.cmaps import VIRIDIS_ALPHA
import torch.nn.functional as F

domains = ["photo", "art_painting", "cartoon", "sketch"]
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
import math
import torch
from tqdm import tqdm
from einops import rearrange
from lib.data_handlers import Load_PACS
import json
import os
import json
from collections import defaultdict, Counter
from overcomplete.visualization.plot_utils import (interpolate_cv2, get_image_dimensions, show)
from overcomplete.visualization.cmaps import VIRIDIS_ALPHA
import torch.nn.functional as F


def calculate_delta(z_sae, y, sae, model):
    # Returns per-image, per-concept change in true-class logit when that concept is masked
    n, w, h, _ = z_sae.shape
    if n == 0:
        nb_concepts = z_sae.shape[-1]
        return torch.zeros((0, nb_concepts), device=z_sae.device)

    pool = SelectAdaptivePool2d(pool_type='avg', flatten=True)

    # Unmasked forward pass
    z_sae = rearrange(z_sae, 'n w h c -> (n w h) c', n=n, h=h, w=w)
    z_recon_flat = sae.decode(z_sae)
    z_recon = rearrange(z_recon_flat, '(n h w) c -> n c h w', n=n, h=h, w=w)
    z_recon = sae.normalizer.denormalize(z_recon)

    
    logits_unmasked = model.classifier(pool(z_recon))
    true_logits_unmasked = logits_unmasked[torch.arange(n, device=device), y]

    # Reshape concept activations per image
    z_sae_img = rearrange(z_sae, '(n h w) c -> n (h w) c', n=n, h=h, w=w)
    concept_max_per_img, _ = z_sae_img.max(dim=1)  # (n, nb_concepts)

    nb_concepts = z_sae.shape[1]
    scores = torch.zeros((n, nb_concepts), device=device)

    # Concepts active at least once in the batch
    if concept_max_per_img.shape[0] == 0:
        return scores

    active_concepts_batch = torch.where(concept_max_per_img.max(dim=0)[0] > 0)[0]

    with torch.no_grad():
        for c in active_concepts_batch:
            active_img_mask = concept_max_per_img[:, c] > 0
            if not active_img_mask.any():
                continue

            original_col = z_sae[:, c].clone()
            z_sae[:, c] = 0

            z_recon_flat_m = sae.decode(z_sae)
            z_recon_m = rearrange(z_recon_flat_m, '(n h w) c -> n c h w', n=n, h=h, w=w)
            z_recon_m = sae.normalizer.denormalize(z_recon_m)

            logits_masked = model.classifier(pool(z_recon_m))
            true_logits_masked = logits_masked[torch.arange(n, device=device), y]

            z_sae[:, c] = original_col

            logit_drop = true_logits_unmasked - true_logits_masked
            scores[active_img_mask, c] = logit_drop[active_img_mask]

    return scores



def calculate_delta_scaled_activations(backbone, sae, rearrange_string, w=14, domains=domains, nb_concepts=7680):
    backbone.eval()
    sae.eval()
    activations = {}
    for cls in range(7):
        
        activations[cls] = {}
        z_d = torch.zeros((len(domains), nb_concepts)).to(device)
        
        ### Added Counts
        counts = torch.zeros((len(domains),)).to(device)  # ADD: track sample counts per domain

        for d, domain in enumerate(domains):
            loader, _ = Load_PACS(domains=[domain])
            for i, batch in enumerate(tqdm(loader)):
                with torch.no_grad():
                    img, y = batch
                    img, y = img.to(device), y.to(device)
                    
                    x = extract_features(backbone, img)
                    x = sae.normalizer(x)
                    x = rearrange(x, rearrange_string)

                    _, heatmaps = sae.encode(x)

                    mask = (y == cls).squeeze().to(device)  # (batch_size,)
                    heatmaps = rearrange(heatmaps, '(n w h) d -> n w h d', w=w, h=w)  # (n, w, h, d)
                    heatmaps_filtered = heatmaps[mask]  # (n_cls, w, h, d)
                    
                    detlas_per_image = calculate_delta(heatmaps_filtered, y[mask], sae, backbone)  # (n, d)
                    activations_per_image = heatmaps_filtered.sum(dim=1).sum(dim=1)  # (n, d)             
                    delta_scaled_activations = detlas_per_image * activations_per_image  # (n, d) 
    
                    z_d[d] += delta_scaled_activations.sum(dim=0)  # Sum over images in the batch 
                    #### Added counter of number of images added
                    counts[d] += mask.sum()

        ## Divide by Safe Count
        safe_counts = counts.clamp(min=1).unsqueeze(1)
        activations[cls] = z_d / safe_counts
        
    return activations


def calculate_invariance(activations, ent_thresh=0.7, act_thresh=0.0, domains=domains, nb_concepts=7680):
    clss = 7
    logs = {"model_invariance" : 0, "final_invariance_per_class": {}, "thresholded_concept_entropies": {}}

    for cls in range(clss):
    
        processed = activations[cls] # * mask
        
        sum_entropy = 0.0
        class_concept_logs = []

        for i in range(nb_concepts):
            if processed[:, i].sum() == 0:
                continue

            #score = processed[:, i] / processed[:, i].sum()
            temp = 1.0
            score = F.softmax(processed[:, i] / temp, dim=0)


            entropy = -1 / torch.log(torch.tensor(len(domains))) * (score * torch.log(score + 1e-12)).sum()


            ## Magnitude Scaling
            # magnitude = processed[:, i].norm()
            # entropy *= magnitude


            if entropy >= ent_thresh and processed[:, i].sum() > act_thresh:
                sum_entropy += entropy

                # --- LOG INDIVIDUAL ENTROPY ---
                class_concept_logs.append({
                    "concept_index": i,
                    "entropy": entropy.item(),
                    "probs": [s.item() for s in score],
                    "scaled_mean_acts": [val.item() for val in processed[:, i]]
                })

        invariance_val = (sum_entropy)
        
        if isinstance(invariance_val, torch.Tensor):
            invariance_float = invariance_val.item()
        else:
            invariance_float = invariance_val # It might already be a float (if sum_entropy was 0.0)

        logs["final_invariance_per_class"][cls] = invariance_float
        logs["model_invariance"] += invariance_float
        logs["thresholded_concept_entropies"][cls] = class_concept_logs
        print(f"Total Thresholded Entropy (INVARIANCE) for class {cls}: {invariance_float}")


    logs["model_invariance"] /= 7
    return logs


def save_json(data, filepath):
    try:
        with open(filepath, 'w') as f:
            json.dump(data, f, indent=4)
        print(f"Successfully saved logs to {filepath}")
    except TypeError as e:
        print(f"Error saving JSON: {e}. Check for non-serializable types (like tensors).")
    except Exception as e:
        print(f"An error occurred: {e}")

In [ ]:
# domains = ["art_painting", "cartoon", "photo"]
domains = ["art_painting", "cartoon", "photo", "sketch"]

for ckpt in SAEs.keys():
    mean_acts = calculate_delta_scaled_activations(backbone=backbones[ckpt].to(device), sae=SAEs[ckpt], rearrange_string="n d w h -> (n w h) d", w=7, nb_concepts=2048*8, domains=domains)
    logs = calculate_invariance(mean_acts, 1.0, 0.0, domains=domains, nb_concepts=2048*8)
    save_json(logs, f"./invariances/RScore4_actthres1_ERM_ResNet_T3_step{ckpt}.json")

In [ ]:
import json
from typing import Dict, Any, Set
from collections import defaultdict, Counter

class ConceptAnalyzer:
    def __init__(self, filepath: str):
        self.filepath = filepath
        self.data = self._load_data()

        self.iou_mode = None
        self.iou_allowed_concepts: Set[int] = set()
        # Populated by calculate_iou(); used by filter_concepts() for the IoU column
        self.concept_class_count: Dict[int, int] = {}

    # ── Data Loading ───────────────────────────────────────────────────────────

    def _load_data(self) -> Dict[str, Any]:
        try:
            with open(self.filepath, 'r') as file:
                return json.load(file)
        except FileNotFoundError:
            print(f"Error: The file {self.filepath} was not found.")
            return {}
        except json.JSONDecodeError:
            print(f"Error: The file {self.filepath} contains invalid JSON.")
            return {}

    # ── IoU Helpers ────────────────────────────────────────────────────────────

    def _get_concept_overlap_mapping(self, min_strength: float = 0.0001) -> Dict[int, set]:
        """
        Returns overlap_buckets: { n_classes -> set of concept_indices }.
        Also populates self.concept_class_count: { concept_idx -> n_classes }.
        """
        concept_data = self.data.get("thresholded_concept_entropies", {})
        if not concept_data:
            return {}

        concept_to_classes: Dict[int, set] = defaultdict(set)
        for class_id_str, concepts in concept_data.items():
            class_id = int(class_id_str)
            for c in concepts:
                idx      = c.get("concept_index")
                strength = sum(c.get("mean_acts", []))
                if idx is not None and strength > min_strength:
                    concept_to_classes[idx].add(class_id)

        # Cache per-concept class count for use in filter_concepts table
        self.concept_class_count = {
            idx: len(classes) for idx, classes in concept_to_classes.items()
        }

        overlap_buckets: Dict[int, set] = defaultdict(set)
        for concept_idx, classes in concept_to_classes.items():
            overlap_buckets[len(classes)].add(concept_idx)

        return overlap_buckets

    def set_iou_mode(self, target_iou: int = None, min_strength: float = 0.0001):
        """
        Restrict filter_concepts to concepts appearing in exactly `target_iou` classes.
        Pass None to disable.
        """
        self.iou_mode = target_iou
        if target_iou is not None:
            buckets = self._get_concept_overlap_mapping(min_strength)
            self.iou_allowed_concepts = buckets.get(target_iou, set())
            print(f"--- IoU Mode Enabled ---")
            print(f"  Target : {target_iou} class(es)")
            print(f"  Locked : {len(self.iou_allowed_concepts)} concept(s)\n")
        else:
            self.iou_allowed_concepts = set()
            print("--- IoU Mode Disabled ---\n")

    # ── Primary Analysis ───────────────────────────────────────────────────────

    def calculate_iou(self, min_strength: float = 0.0001) -> Dict[int, set]:
        """
        For each concept, counts how many classes it appears in and prints a
        distribution table. Populates self.concept_class_count as a side-effect
        so filter_concepts can show the IoU column without recomputing.
        """
        if not self.data:
            print("No data loaded.")
            return {}

        concept_data = self.data.get("thresholded_concept_entropies", {})
        if not concept_data:
            print("No 'thresholded_concept_entropies' found in the data.")
            return {}

        total_classes    = len(concept_data)
        overlap_buckets  = self._get_concept_overlap_mapping(min_strength)   # also fills concept_class_count
        total_concepts   = sum(len(v) for v in overlap_buckets.values())
        bar_width        = 30

        print(f"{'═'*62}")
        print(f"  CONCEPT CLASS OVERLAP  (min_strength > {min_strength})")
        print(f"{'═'*62}")
        print(f"  {'Classes':<10} {'# Concepts':>12}   {'Distribution'}")
        print(f"  {'─'*58}")

        for n in range(1, total_classes + 1):
            bucket     = overlap_buckets.get(n, set())
            count      = len(bucket)
            proportion = count / total_concepts if total_concepts else 0
            bar        = "█" * int(proportion * bar_width)
            label      = "class only" if n == 1 else "classes"
            print(f"  {n} {label:<12} {count:>8}   {bar} {proportion*100:.1f}%")

        print(f"  {'─'*58}")
        print(f"  {'Total':<22} {total_concepts:>8}")
        print(f"{'═'*62}\n")

        for n in range(1, total_classes + 1):
            bucket = overlap_buckets.get(n, set())
            if bucket:
                print(f"  Concepts in exactly {n} class(es) [{len(bucket)}]:")
                sorted_concepts = sorted(bucket)
                for i in range(0, len(sorted_concepts), 10):
                    print(f"    {sorted_concepts[i:i+10]}")
                print()

        return dict(overlap_buckets)

    def filter_concepts(
        self,
        entropy_min:           float = None,
        entropy_max:           float = None,
        discrimination_min:    float = None,
        discrimination_max:    float = None,
        mean_acts_sum_min:     float = None,
        mean_acts_sum_max:     float = None,
        activation_count_min:  int   = None,
        activation_count_max:  int   = None,
    ):
        """
        Filters concepts across all classes and prints a table.
        If calculate_iou() has been called beforehand, an 'IoU (#cls)' column
        showing how many classes each concept appears in is included automatically.
        """
        if not self.data:
            print("No data loaded.")
            return

        concept_data = self.data.get("thresholded_concept_entropies", {})
        if not concept_data:
            print("No 'thresholded_concept_entropies' found in the data.")
            return

        has_iou = bool(self.concept_class_count)   # True if calculate_iou() was called

        # ── Active filter summary ──────────────────────────────────────────────
        print("--- Active Filters ---")
        if self.iou_mode is not None:        print(f"  IoU Mode          == {self.iou_mode} class(es)")
        if entropy_min is not None:          print(f"  Entropy           >= {entropy_min}")
        if entropy_max is not None:          print(f"  Entropy           <= {entropy_max}")
        if discrimination_min is not None:   print(f"  Discrimination    >= {discrimination_min}")
        if discrimination_max is not None:   print(f"  Discrimination    <= {discrimination_max}")
        if mean_acts_sum_min is not None:    print(f"  Mean Acts Sum     >= {mean_acts_sum_min}")
        if mean_acts_sum_max is not None:    print(f"  Mean Acts Sum     <= {mean_acts_sum_max}")
        if activation_count_min is not None: print(f"  Activation Count  >= {activation_count_min}")
        if activation_count_max is not None: print(f"  Activation Count  <= {activation_count_max}")
        if not has_iou:
            print("  (IoU column hidden — run calculate_iou() first to enable it)")
        print()

        total_matches = 0

        for class_id, concepts in concept_data.items():
            matched = []
            for c in concepts:
                concept_idx      = c.get("concept_index")
                entropy          = c.get("entropy", 0)
                discrimination   = c.get("discrimination_score", 0)
                mean_acts_sum    = sum(c.get("scaled_mean_acts", []))
                activation_count = c.get("activation_count", 0)

                if self.iou_mode is not None and concept_idx not in self.iou_allowed_concepts:
                    continue
                if entropy_min is not None          and entropy < entropy_min:                   continue
                if entropy_max is not None          and entropy > entropy_max:                   continue
                if discrimination_min is not None   and discrimination < discrimination_min:     continue
                if discrimination_max is not None   and discrimination > discrimination_max:     continue
                if mean_acts_sum_min is not None    and mean_acts_sum < mean_acts_sum_min:       continue
                if mean_acts_sum_max is not None    and mean_acts_sum > mean_acts_sum_max:       continue
                if activation_count_min is not None and activation_count < activation_count_min: continue
                if activation_count_max is not None and activation_count > activation_count_max: continue

                matched.append(c)

            if matched:
                print(f"Class {class_id}: {len(matched)} match(es)")

                # ── Header ────────────────────────────────────────────────────
                if has_iou:
                    print(f"  {'Concept':<10} {'Entropy':<10} {'Discrimination':<18} {'Mean Acts Sum':<16} {'Act. Count':<12} {'IoU (#cls)'}")
                    print(f"  {'-'*82}")
                else:
                    print(f"  {'Concept':<10} {'Entropy':<10} {'Discrimination':<18} {'Mean Acts Sum':<16} {'Act. Count'}")
                    print(f"  {'-'*68}")

                # ── Rows ──────────────────────────────────────────────────────
                for c in matched:
                    cidx = c.get("concept_index")
                    row = (
                        f"  {cidx:<10} "
                        f"{c.get('entropy', 0):<10.4f} "
                        f"{c.get('discrimination_score', 0):<18.6f} "
                        f"{sum(c.get('mean_acts', [])):<16.2f} "
                        f"{c.get('activation_count', 0):<12}"
                    )
                    if has_iou:
                        n_classes = self.concept_class_count.get(cidx, 0)
                        row += f" {n_classes}"
                    print(row)
                print()
                total_matches += len(matched)

        print(f"Total matching concepts across all classes: {total_matches}")

In [ ]:
import os
import torch
import random
import numpy as np
from PIL import Image
from tqdm import tqdm
from einops import rearrange
import matplotlib.pyplot as plt
from torchvision import transforms
from lib.data_handlers import Load_PACS
from overcomplete.visualization.cmaps import VIRIDIS_ALPHA
from overcomplete.visualization.plot_utils import (interpolate_cv2, get_image_dimensions, show)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.cuda.empty_cache()

dirs = {
    "art_painting"  : r"C:\Users\sproj_ha\Desktop\DomainBed\domainbed\data\PACS\art_painting",
    "sketch"        : r"C:\Users\sproj_ha\Desktop\DomainBed\domainbed\data\PACS\sketch",
    "photo"         : r"C:\Users\sproj_ha\Desktop\DomainBed\domainbed\data\PACS\photo",
    "cartoon"       : r"C:\Users\sproj_ha\Desktop\DomainBed\domainbed\data\PACS\cartoon",
}

def visualize_class_on_concept(concept, class_idx, model, sae, rearrange_string, w=14, domain_roots=dirs, save_dir=None, n_images=None):
    

    model.eval()
    sae.eval()

    domain_top_images = {}  # domain -> list of (heatmap_sum, img_tensor, heatmap)
    
    # 1. Build the global sorted class list EXACTLY like the dataloader
    all_classes = set()
    for d_path in domain_roots.values():
        for entry in os.scandir(d_path):
            if entry.is_dir():
                all_classes.add(entry.name)
    sorted_classes = sorted(list(all_classes))
    
    # 2. Map the index to the actual string name
    target_class_name = sorted_classes[class_idx]

    for domain, dir_path in domain_roots.items():
        # 3. Safely build the path using the string name
        class_dir = os.path.join(dir_path, target_class_name)
        
        if not os.path.exists(class_dir):
            continue
            
        images = [Image.open(os.path.join(class_dir, path)) for path in os.listdir(class_dir)]

        if n_images is not None and n_images < len(images):
            images = random.sample(images, n_images)

        results = []  # (heatmap_sum, img_tensor, heatmap)

        for i, img in enumerate(images):
            with torch.no_grad():
                img = img.convert("RGB")
                transform = transforms.Compose([
                        transforms.Resize(256),
                        transforms.CenterCrop(224),
                        transforms.ToTensor(),
                        transforms.Normalize(mean=[0.485, 0.456, 0.406],
                                            std=[0.229, 0.224, 0.225])
                ])

                img_tensor = transform(img).unsqueeze(dim=0).to(device) # Don't forget to send to device!

                x = extract_features(model, img_tensor)
                x = sae.normalizer(x)
                
                x = rearrange(x, rearrange_string)
                
                _, z = sae.encode(x)
                
                # FIX 2: Use dynamic spatial dimensions
                z = rearrange(z, '(n w h) d -> n w h d', w=w, h=w)
                
                width, height = img_tensor.shape[-1], img_tensor.shape[-2]
                
                # FIX 3: Isolate the specific 2D image, detach, move to CPU, and convert to numpy
                heatmap_2d = z[0, :, :, concept].detach().cpu().numpy()
                
                heatmap = interpolate_cv2(heatmap_2d, (width, height))
                heatmap_sum = heatmap.sum()

                if heatmap_sum > 0:
                    results.append((heatmap_sum, img_tensor.cpu(), heatmap)) # Move img_tensor back to CPU for storage/plotting

        # Sort by activation and keep top 8
        results.sort(key=lambda x: x[0], reverse=True)
        domain_top_images[domain] = results[:8]

    # Build grid: rows = domains, cols = top-8 images
    domains = list(domain_top_images.keys())
    n_domains = len(domains)
    n_cols = 8

    fig, axes = plt.subplots(n_domains, n_cols, figsize=(n_cols * 2, n_domains * 2))

    # Ensure axes is always 2D
    if n_domains == 1:
        axes = axes[np.newaxis, :]
    if n_cols == 1:
        axes = axes[:, np.newaxis]

    for row, domain in enumerate(domains):
        top = domain_top_images[domain]
        for col in range(n_cols):
            ax = axes[row, col]
            ax.axis("off")
            if col < len(top):
                _, img_tensor, heatmap = top[col]
                # Convert image tensor to HWC numpy for display
                show(img_tensor, ax=ax)
                show(heatmap, ax=ax, cmap=VIRIDIS_ALPHA, alpha=1.0)
            if col == 0:
                ax.set_title(domain, fontsize=8, loc='left', pad=2)

    plt.suptitle(f"Class {target_class_name} — Concept {concept} | Top 8 Activations", fontsize=11, y=1.01)
    plt.tight_layout()

    if save_dir is not None:
        os.makedirs(save_dir, exist_ok=True)
        plt.savefig(os.path.join(save_dir, f"Class{class_idx}_Concept{concept}_Grid.png"), bbox_inches="tight")
        plt.close()
    else:
        plt.show()

In [ ]:
analyzer = ConceptAnalyzer(f"./invariances/RScore_ERM_ResNet_T3_step3300.json")
analyzer.filter_concepts(
    entropy_max=None,
    entropy_min=0.9,
    mean_acts_sum_min=1.0,
    mean_acts_sum_max=None,
)

In [ ]:
import json
from typing import List, Tuple, Optional

def _in_any_interval(value: float, intervals: Optional[List[Tuple[float, float]]]) -> bool:
    """Return True if value falls within ANY of the given [min, max] intervals.
    If intervals is None or empty, the check is skipped (always passes)."""
    if not intervals:
        return True
    return any(lo <= value <= hi for lo, hi in intervals)


def masked_accuracy_without_label_leakage(
    model,
    sae,
    dataloader,
    json_filepath: str,
    generalization_score_intervals: Optional[List[Tuple[float, float]]] = None,
    entropy_intervals:              Optional[List[Tuple[float, float]]] = None,
    discrimination_intervals:       Optional[List[Tuple[float, float]]] = None,
    num_classes: int = 7,
    nb_concepts: int = 7680,
    device: str = 'cuda',
) -> dict:

    # ------------------------------------------------------------------ #
    # 1. Load JSON and build a single global concept mask                  #
    # ------------------------------------------------------------------ #
    with open(json_filepath, 'r') as f:
        data = json.load(f)

    concept_data = data.get("thresholded_concept_entropies", {})

    global_indices = set()

    for class_id_str, concepts in concept_data.items():
        for c in concepts:
            entropy        = c.get("entropy", 0.0)
            discrimination = c.get("discrimination_score", 0.0)
            gen_score      = entropy * discrimination

            if not _in_any_interval(gen_score,      generalization_score_intervals): continue
            if not _in_any_interval(entropy,         entropy_intervals):              continue
            if not _in_any_interval(discrimination,  discrimination_intervals):       continue

            global_indices.add(c["concept_index"])

    global_concept_mask = torch.zeros(nb_concepts, dtype=torch.bool, device=device)
    if global_indices:
        global_concept_mask[list(global_indices)] = True

    # ------------------------------------------------------------------ #
    # 2. Single-pass: baseline and masked accuracy together                #
    # ------------------------------------------------------------------ #
    model.eval()
    pool = SelectAdaptivePool2d(pool_type='avg', flatten=True)

    correct_baseline = torch.zeros(num_classes, device=device)
    correct_masked   = torch.zeros(num_classes, device=device)
    total_per_class  = torch.zeros(num_classes, device=device)

    with torch.no_grad():
        for x, y in tqdm(dataloader, desc="Masked vs Baseline Accuracy"):
            x, y = x.to(device), y.to(device)
            n = x.size(0)

            z_raw  = extract_features(model, x)
            z_norm = sae.normalizer(z_raw)
            _, _, h, w = z_norm.shape

            z_flat   = rearrange(z_norm, "n c h w -> (n h w) c")
            _, z_sae = sae.encode(z_flat)

            # --- Baseline (no masking) ---
            z_recon_flat   = sae.decode(z_sae)
            z_recon        = rearrange(z_recon_flat, '(n h w) c -> n c h w', n=n, h=h, w=w)
            z_recon        = sae.normalizer.denormalize(z_recon)
            preds_baseline = model.classifier(pool(z_recon)).argmax(dim=1)

            # --- Masked pass: single global mask, no label needed ---
            z_sae_masked = z_sae.clone()
            z_sae_masked[:, global_concept_mask] = 0.0

            z_recon_flat_m = sae.decode(z_sae_masked)
            z_recon_m      = rearrange(z_recon_flat_m, '(n h w) c -> n c h w', n=n, h=h, w=w)
            z_recon_m      = sae.normalizer.denormalize(z_recon_m)
            preds_masked   = model.classifier(pool(z_recon_m)).argmax(dim=1)

            # --- Accumulate ---
            for cls in range(num_classes):
                cls_mask = (y == cls)
                correct_baseline[cls] += (preds_baseline[cls_mask] == y[cls_mask]).sum()
                correct_masked[cls]   += (preds_masked[cls_mask]   == y[cls_mask]).sum()
                total_per_class[cls]  += cls_mask.sum()

    # ------------------------------------------------------------------ #
    # 3. Compute metrics                                                   #
    # ------------------------------------------------------------------ #
    per_class_baseline = (correct_baseline / total_per_class.clamp(min=1)).cpu()
    per_class_masked   = (correct_masked   / total_per_class.clamp(min=1)).cpu()
    overall_baseline   = (correct_baseline.sum() / total_per_class.sum()).item()
    overall_masked     = (correct_masked.sum()    / total_per_class.sum()).item()

    return {
        "per_class_baseline_accuracy": {cls: per_class_baseline[cls].item() for cls in range(num_classes)},
        "per_class_masked_accuracy":   {cls: per_class_masked[cls].item()   for cls in range(num_classes)},
        "overall_baseline_accuracy":   overall_baseline,
        "overall_masked_accuracy":     overall_masked,
        "overall_delta":               overall_masked - overall_baseline,
        "total_concepts_masked":       len(global_indices),
        "masked_concept_indices":      sorted(global_indices),
    }

In [ ]:
domains = ["art_painting", "cartoon", "photo", "sketch"]


ckpt = 3300

all_domain_results = {}

for domain in domains:
    val_loader, _ = Load_PACS(domains=[domain])

    results = masked_accuracy_without_label_leakage(
        model=backbones[ckpt].to(device),
        sae=SAEs[ckpt],
        dataloader=val_loader,
        json_filepath=f"./invariances/RScore_ERM_ResNet_T3_step{ckpt}.json",
        entropy_intervals=[(0.0, 0.9)],  # Only concepts with entropy >= 0.9
        num_classes=7,
        nb_concepts=2048*8,
        device='cuda',
    )

    all_domain_results[domain] = results



# ── Cross-Domain Summary ──────────────────────────────────────────────────────
print(f"\n{'═'*62}")
print(f"  SUMMARY")
print(f"{'═'*62}")
print(f"  {'Domain':<20} {'Baseline':>10} {'Masked':>10} {'Δ':>8}")
print(f"  {'─'*54}")
for domain, results in all_domain_results.items():
    b    = results["overall_baseline_accuracy"] * 100
    m    = results["overall_masked_accuracy"]   * 100
    d    = m - b
    sign = "+" if d >= 0 else ""
    print(f"  {domain.replace('_', ' ').title():<20} {b:>9.2f}% {m:>9.2f}% {sign}{d:>6.2f}%")
print(f"  {'─'*54}")

# Averages across domains
avg_b = sum(r["overall_baseline_accuracy"] for r in all_domain_results.values()) / len(all_domain_results) * 100
avg_m = sum(r["overall_masked_accuracy"]   for r in all_domain_results.values()) / len(all_domain_results) * 100
avg_d = avg_m - avg_b
sign  = "+" if avg_d >= 0 else ""
print(f"  {'Average':<20} {avg_b:>9.2f}% {avg_m:>9.2f}% {sign}{avg_d:>6.2f}%")
print(f"{'═'*62}\n")

In [ ]:
visualize_class_on_concept(11588, 3, backbones[2100].to(device), sae=SAEs[2100], rearrange_string='n c w h -> (n w h) c', w=7, save_dir=None, n_images=None, domain_roots=dirs) 

## Per Class Accuracies

In [ ]:
import json
from typing import List, Tuple, Optional

def _in_any_interval(value: float, intervals: Optional[List[Tuple[float, float]]]) -> bool:
    """Return True if value falls within ANY of the given [min, max] intervals.
    If intervals is None or empty, the check is skipped (always passes)."""
    if not intervals:
        return True
    return any(lo <= value <= hi for lo, hi in intervals)


def masked_accuracy_without_label_leakage(
    model,
    sae,
    dataloader,
    json_filepath: str,
    generalization_score_intervals: Optional[List[Tuple[float, float]]] = None,
    entropy_intervals:              Optional[List[Tuple[float, float]]] = None,
    discrimination_intervals:       Optional[List[Tuple[float, float]]] = None,
    num_classes: int = 7,
    nb_concepts: int = 7680,
    device: str = 'cuda',
) -> dict:

    # ------------------------------------------------------------------ #
    # 1. Load JSON and build a single global concept mask                  #
    # ------------------------------------------------------------------ #
    with open(json_filepath, 'r') as f:
        data = json.load(f)

    concept_data = data.get("thresholded_concept_entropies", {})

    global_indices = set()

    for class_id_str, concepts in concept_data.items():
        for c in concepts:
            entropy        = c.get("entropy", 0.0)
            discrimination = c.get("discrimination_score", 0.0)
            gen_score      = entropy * discrimination

            if not _in_any_interval(gen_score,      generalization_score_intervals): continue
            if not _in_any_interval(entropy,         entropy_intervals):              continue
            if not _in_any_interval(discrimination,  discrimination_intervals):       continue

            global_indices.add(c["concept_index"])

    global_concept_mask = torch.zeros(nb_concepts, dtype=torch.bool, device=device)
    if global_indices:
        global_concept_mask[list(global_indices)] = True

    # ------------------------------------------------------------------ #
    # 2. Single-pass: baseline and masked accuracy together                #
    # ------------------------------------------------------------------ #
    model.eval()
    pool = SelectAdaptivePool2d(pool_type='avg', flatten=True)

    correct_baseline = torch.zeros(num_classes, device=device)
    correct_masked   = torch.zeros(num_classes, device=device)
    total_per_class  = torch.zeros(num_classes, device=device)

    with torch.no_grad():
        for x, y in tqdm(dataloader, desc="Masked vs Baseline Accuracy"):
            x, y = x.to(device), y.to(device)
            n = x.size(0)

            z_raw  = extract_features(model, x)
            z_norm = sae.normalizer(z_raw)
            _, _, h, w = z_norm.shape

            z_flat   = rearrange(z_norm, "n c h w -> (n h w) c")
            _, z_sae = sae.encode(z_flat)

            # --- Baseline (no masking) ---
            z_recon_flat   = sae.decode(z_sae)
            z_recon        = rearrange(z_recon_flat, '(n h w) c -> n c h w', n=n, h=h, w=w)
            z_recon        = sae.normalizer.denormalize(z_recon)
            preds_baseline = model.classifier(pool(z_recon)).argmax(dim=1)

            # --- Masked pass: single global mask, no label needed ---
            z_sae_masked = z_sae.clone()
            z_sae_masked[:, global_concept_mask] = 0.0

            z_recon_flat_m = sae.decode(z_sae_masked)
            z_recon_m      = rearrange(z_recon_flat_m, '(n h w) c -> n c h w', n=n, h=h, w=w)
            z_recon_m      = sae.normalizer.denormalize(z_recon_m)
            preds_masked   = model.classifier(pool(z_recon_m)).argmax(dim=1)

            # --- Accumulate ---
            for cls in range(num_classes):
                cls_mask = (y == cls)
                correct_baseline[cls] += (preds_baseline[cls_mask] == y[cls_mask]).sum()
                correct_masked[cls]   += (preds_masked[cls_mask]   == y[cls_mask]).sum()
                total_per_class[cls]  += cls_mask.sum()

    # ------------------------------------------------------------------ #
    # 3. Compute metrics                                                   #
    # ------------------------------------------------------------------ #
    per_class_baseline = (correct_baseline / total_per_class.clamp(min=1)).cpu()
    per_class_masked   = (correct_masked   / total_per_class.clamp(min=1)).cpu()
    overall_baseline   = (correct_baseline.sum() / total_per_class.sum()).item()
    overall_masked     = (correct_masked.sum()    / total_per_class.sum()).item()

    return {
        "per_class_baseline_accuracy": {cls: per_class_baseline[cls].item() for cls in range(num_classes)},
        "per_class_masked_accuracy":   {cls: per_class_masked[cls].item()   for cls in range(num_classes)},
        "overall_baseline_accuracy":   overall_baseline,
        "overall_masked_accuracy":     overall_masked,
        "overall_delta":               overall_masked - overall_baseline,
        "total_concepts_masked":       len(global_indices),
        "masked_concept_indices":      sorted(global_indices),
    }

In [ ]:
domains = ["art_painting", "cartoon", "photo", "sketch"]

all_domain_results = {}

for domain in domains:
    val_loader, _ = Load_PACS(domains=[domain])

    results = masked_accuracy_without_label_leakage(
        model=backbones[3300].to(device),
        sae=SAEs[3300],
        dataloader=val_loader,
        json_filepath="./invariances/uGEN_ERM_ResNet_T3_step3300.json",
        num_classes=7,
        nb_concepts=2048*8,
        device='cuda',
        entropy_intervals=None,
        discrimination_intervals=[(-1.0, -0.001), (0.001, 1.0)],
        generalization_score_intervals=None,   
    )

    all_domain_results[domain] = results



# ── Cross-Domain Summary ──────────────────────────────────────────────────────
print(f"\n{'═'*62}")
print(f"  SUMMARY")
print(f"{'═'*62}")
print(f"  {'Domain':<20} {'Baseline':>10} {'Masked':>10} {'Δ':>8}")
print(f"  {'─'*54}")
for domain, results in all_domain_results.items():
    b    = results["overall_baseline_accuracy"] * 100
    m    = results["overall_masked_accuracy"]   * 100
    d    = m - b
    sign = "+" if d >= 0 else ""
    print(f"  {domain.replace('_', ' ').title():<20} {b:>9.2f}% {m:>9.2f}% {sign}{d:>6.2f}%")
print(f"  {'─'*54}")

# Averages across domains
avg_b = sum(r["overall_baseline_accuracy"] for r in all_domain_results.values()) / len(all_domain_results) * 100
avg_m = sum(r["overall_masked_accuracy"]   for r in all_domain_results.values()) / len(all_domain_results) * 100
avg_d = avg_m - avg_b
sign  = "+" if avg_d >= 0 else ""
print(f"  {'Average':<20} {avg_b:>9.2f}% {avg_m:>9.2f}% {sign}{avg_d:>6.2f}%")
print(f"{'═'*62}\n")

In [ ]:
import torch
import json
from tqdm import tqdm

def calculate_invariance_metrics(
    backbone, 
    checkpoint_step, 
    domains=["art_painting", "cartoon", "photo", "sketch"],
    num_classes=7,
    device='cuda'
):
    """
    Creates a summary table: 7 classes (rows) vs 4 Domain Accuracies + Mean Entropy (cols).
    """
    backbone.to(device)
    backbone.eval()
    pool = SelectAdaptivePool2d(pool_type='avg', flatten=True)




    # 1. Load Entropy data from the JSON file
    json_path = f"./invariances/RScore_ERM_ResNet_T3_step{checkpoint_step}.json"
    try:
        with open(json_path, 'r') as f:
            data = json.load(f)
    except FileNotFoundError:
        print(f"Error: {json_path} not found.")
        return

    concept_data = data.get("thresholded_concept_entropies", {})
    
    # Calculate Mean Entropy per class
    mean_entropies = {}
    for cls_idx in range(num_classes):
        cls_str = str(cls_idx)
        concepts = concept_data.get(cls_str, [])
        if concepts:
            avg_ent = sum(c.get("entropy", 0.0) for c in concepts) / len(concepts)
        else:
            avg_ent = 0.0
        mean_entropies[cls_idx] = avg_ent

    # 2. Calculate Per-Class Accuracy for each domain
    # Store as: results[domain][class_id] = accuracy
    domain_results = {d: {} for d in domains}

    for domain in domains:
        # Assuming Load_PACS returns (dataloader, _)
        val_loader, _ = Load_PACS(domains=[domain])
        
        correct = torch.zeros(num_classes, device=device)
        total = torch.zeros(num_classes, device=device)

        with torch.no_grad():
            for x, y in tqdm(val_loader, desc=f"Processing {domain}", leave=False):
                x, y = x.to(device), y.to(device)
                
                z = extract_features(backbone, x)
                preds = backbone.classifier(pool(z)).argmax(dim=1)
                
                
                for cls in range(num_classes):
                    mask = (y == cls)
                    correct[cls] += (preds[mask] == y[mask]).sum()
                    total[cls] += mask.sum()

        per_class_acc = (correct / total.clamp(min=1)).cpu().numpy()
        for cls_idx, acc in enumerate(per_class_acc):
            domain_results[domain][cls_idx] = acc * 100

    # 3. Print the Clean Table
    header_domains = " | ".join([f"{d[:8]:>10}" for d in domains])
    print(f"\n{'═'*90}")
    print(f" INVARIANCES REPORT - STEP {checkpoint_step}")
    print(f"{'═'*90}")
    print(f"{'Class':<8} | {header_domains} | {'Mean Entropy':>12}")
    print(f"{'─'*90}")

    for cls in range(num_classes):
        row_str = f"Class {cls:<2} | "
        for domain in domains:
            acc = domain_results[domain][cls]
            row_str += f"{acc:>10.2f}% | "
        
        ent = mean_entropies[cls]
        row_str += f"{ent:>12.6f}"
        print(row_str)

    print(f"{'═'*90}\n")

# --- Usage ---
calculate_invariance_metrics(backbones[3300], 3300)
calculate_invariance_metrics(backbones[2100], 2100)

## Effect of Logits

In [ ]:
def calculate_discrimination_scores_with_logits(model, sae, dataloader, num_classes=7, nb_concepts=7680, device='cuda'):
    """
    ASSUMPTIONS:
    1. Score = logit_unmasked(y_true) - logit_masked(y_true).
       - Positive: Concept supports the ground truth.
       - Negative: Concept acted as a distractor/caused an error.
    2. Scores are only computed/aggregated for images where the concept's activation > 0.
    3. Aggregated over the entire dataset returning shape: (num_classes, nb_concepts).
    """
    model.eval()
    sae.eval()
    pool = SelectAdaptivePool2d(pool_type='avg', flatten=True)

    accumulated_scores = torch.zeros((num_classes, nb_concepts), device=device)
    activation_counts  = torch.zeros((num_classes, nb_concepts), device=device)

    for x, y in tqdm(dataloader, desc="Calculating Concept Discrimination (Logits)"):
        x, y = x.to(device), y.to(device)
        n = x.size(0)

        with torch.no_grad():
            # --- Base Unmasked Pass ---
            z_raw  = extract_features(model, x)
            z_norm = sae.normalizer(z_raw)
            _, _, h, w = z_norm.shape

            z_flat = rearrange(z_norm, "n c h w -> (n h w) c")
            _, z_sae = sae.encode(z_flat)

            z_recon_flat = sae.decode(z_sae)
            z_recon      = rearrange(z_recon_flat, '(n h w) c -> n c h w', n=n, h=h, w=w)
            z_recon      = sae.normalizer.denormalize(z_recon)

            logits_unmasked  = model.classifier(pool(z_recon))
            logit_true_unmasked = logits_unmasked[torch.arange(n), y]  # (n,) — raw logit, no softmax
            prob_true_unmasked = torch.softmax(logits_unmasked, dim=1)[torch.arange(n), y]  # (n,) — probability after softmax
            # --- Fast Masking Setup ---
            z_sae_img = rearrange(z_sae, '(n h w) c -> n (h w) c', n=n, h=h, w=w)
            concept_max_per_img, _ = z_sae_img.max(dim=1)  # (n, nb_concepts)

            active_concepts_batch = torch.where(concept_max_per_img.max(dim=0)[0] > 0)[0]

            for c in active_concepts_batch:
                active_img_mask = concept_max_per_img[:, c] > 0

                if not active_img_mask.any():
                    continue

                original_col = z_sae[:, c].clone()
                z_sae[:, c]  = 0

                z_recon_flat_m = sae.decode(z_sae)
                z_recon_m      = rearrange(z_recon_flat_m, '(n h w) c -> n c h w', n=n, h=h, w=w)
                z_recon_m      = sae.normalizer.denormalize(z_recon_m)

                logits_masked       = model.classifier(pool(z_recon_m))
                logit_true_masked   = logits_masked[torch.arange(n), y]  # (n,) — raw logit, no softmax
                prob_true_masked  = torch.softmax(logits_masked, dim=1)[torch.arange(n), y]  # (n,) — probability after softmax

                z_sae[:, c] = original_col

                #score_drop = logit_true_unmasked - logit_true_masked  # (n,)
                score_drop = prob_true_unmasked - prob_true_masked

                active_classes = y[active_img_mask]
                active_drops   = score_drop[active_img_mask]

                accumulated_scores[:, c].scatter_add_(0, active_classes, active_drops)
                activation_counts[:, c].scatter_add_(0, active_classes, torch.ones_like(active_drops))

    final_discrimination_scores = accumulated_scores / activation_counts.clamp(min=1)

    return final_discrimination_scores, activation_counts




import json

def combine_scores_to_json(existing_json_path, output_json_path, scores, counts):
    """
    Loads the existing invariance JSON, injects discrimination scores 
    AND activation counts for each concept, and saves the combined data.
    Concepts missing from the JSON are appended with default attribute values.
    """
    try:
        with open(existing_json_path, 'r') as f:
            data = json.load(f)
    except Exception as e:
        print(f"Error loading JSON: {e}")
        return

    concept_data = data.get("thresholded_concept_entropies", {})

    num_classes, num_concepts = scores.shape

    for class_id in range(num_classes):
        class_id_str = str(class_id)

        # Ensure this class key exists in the JSON
        if class_id_str not in concept_data:
            concept_data[class_id_str] = []

        existing_concepts = concept_data[class_id_str]

        # Build a lookup of concept_index -> concept dict for fast access
        existing_index_map = {
            c["concept_index"]: c
            for c in existing_concepts
            if "concept_index" in c
        }

        for c_idx in range(num_concepts):
            disc_score = float(scores[class_id, c_idx].item())
            act_count  = int(counts[class_id, c_idx].item())

            if c_idx in existing_index_map:
                # Update the existing concept entry
                concept = existing_index_map[c_idx]
                concept["discrimination_score"] = disc_score if act_count > 0 else 0.0
                concept["activation_count"]     = act_count
            else:
                
                if act_count > 0 and disc_score != 0.0:
                    
                    new_concept = {
                        "concept_index":        c_idx,
                        "entropy":              -1,
                        "scores":               [0, 0, 0],
                        "mean_acts":            [0, 0, 0],
                        "discrimination_score": disc_score if act_count > 0 else 0.0,
                        "activation_count":     act_count,
                        "generalization_score": 0.0,
                    }
                    existing_concepts.append(new_concept)

    with open(output_json_path, 'w') as f:
        json.dump(data, f, indent=4)

    print(f"Successfully created combined JSON at: {output_json_path}")




import json
import matplotlib.pyplot as plt

def plot_concept_quality(combined_json_filepath, class_to_plot=None, min_occurrences=0):
    """
    Reads the combined JSON and plots Discrimination (X) vs. Invariance/Entropy (Y).
    If class_to_plot is provided, it only plots that specific class.
    Only plots concepts that fired in at least `min_occurrences` images.
    X-axis is fixed to [-1, 1] and Y-axis is fixed to [0, 1].
    """
    try:
        with open(combined_json_filepath, 'r') as f:
            data = json.load(f)
    except Exception as e:
        print(f"Error loading Combined JSON: {e}")
        return

    concept_data = data.get("thresholded_concept_entropies", {})

    plt.figure(figsize=(12, 8))
    colors = plt.cm.tab10.colors  

    if class_to_plot is not None:
        class_to_plot = str(class_to_plot)

    for class_id_str, concepts in concept_data.items():
        if class_to_plot is not None and class_id_str != class_to_plot:
            continue

        x_vals = [] 
        y_vals = [] 
        
        for concept in concepts:
            entropy = concept.get("entropy")
            disc_score = concept.get("discrimination_score")
            act_count = concept.get("activation_count", 0) # Safely default to 0 if missing

            # Check metrics AND that it meets the minimum occurrence threshold
            if (entropy is not None and 
                disc_score is not None and 
                disc_score != 0.0 and 
                act_count >= min_occurrences):
                
                x_vals.append(disc_score)
                y_vals.append(entropy)

        if x_vals and y_vals:
            plt.scatter(
                x_vals, 
                y_vals, 
                label=f"Class {class_id_str}", 
                color=colors[int(class_id_str) % len(colors)], 
                alpha=0.7,
                edgecolors='w',
                linewidth=0.5
            )

    title_suffix = f" (Class {class_to_plot})" if class_to_plot is not None else " (All Classes)"
    filter_suffix = f" | Min Occurrences: {min_occurrences}" if min_occurrences > 0 else ""
    plt.title(f"Discrimination vs. Invariance{title_suffix}{filter_suffix}", fontsize=16, fontweight='bold')
    
    plt.xlabel("Discrimination Score (Drop in Ground Truth Confidence)", fontsize=12)
    plt.ylabel("Invariance Score (Entropy across Domains)", fontsize=12)
    
    plt.xlim(-0.1, 0.1)
    plt.ylim(0, 1)
    
    plt.axvline(0, color='black', linewidth=1.5, linestyle='--')
    
    plt.grid(True, linestyle='--', alpha=0.5)
    
    if plt.gca().get_legend_handles_labels()[0]:
        plt.legend(title="Classes", bbox_to_anchor=(1.05, 1), loc='upper left')
        
    plt.tight_layout()
    plt.show()

In [ ]:
domains = ["art_painting", "cartoon", "photo"]
loader, _ = Load_PACS(domains=domains)


for ckpt in backbones.keys():
    scores, counts = calculate_discrimination_scores_with_logits(
        model=backbones[ckpt].to(device),
        sae=SAEs[ckpt],
        dataloader=loader,
        num_classes=7,
        nb_concepts=2048 * 8,
        device=device
    )

    combine_scores_to_json(
        existing_json_path=os.path.join(f"./invariances/RScore_ERM_ResNet_T3_step{ckpt}.json"), 
        output_json_path=os.path.join(f"./invariances/RDProb_ERM_ResNet_T3_step{ckpt}.json"), 
        scores=scores, 
        counts=counts
    )

In [ ]:
# 2. Plot from the single file
plot_concept_quality(r"./invariances/RD_ERM_ResNet_T3_step3300.json", class_to_plot=None, min_occurrences=10)
plot_concept_quality(r"./invariances/RD_ERM_ResNet_T3_step2100.json", class_to_plot=None, min_occurrences=10)

In [ ]:
# 2. Plot from the single file
plot_concept_quality(r"./invariances/RScore_ERM_ResNet_T3_step3300.json", class_to_plot=None, min_occurrences=10)

## DG Collapse

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from einops import rearrange
from itertools import cycle

pool = SelectAdaptivePool2d(pool_type='avg', flatten=False)


class GumbelConceptMask(nn.Module):
    def __init__(self, nb_concepts: int, init_logit: float = 2.0):
        super().__init__()
        # We model p(keep) via a 2-class Gumbel-Softmax: [logit_zero, logit_one]
        self.logits = nn.Parameter(
            torch.stack([
                torch.full((nb_concepts,), -init_logit),   # logit for 0
                torch.full((nb_concepts,),  init_logit),   # logit for 1
            ], dim=-1)  # (nb_concepts, 2)
        )

    def forward(self, tau: float = 1.0, hard: bool = True) -> torch.Tensor:
        # logits: (nb_concepts, 2)
        mask_soft = F.gumbel_softmax(self.logits, tau=tau, hard=hard, dim=-1)
        return mask_soft[..., 1]   # (nb_concepts,)

    @property
    def keep_probs(self) -> torch.Tensor:
        return torch.softmax(self.logits, dim=-1)[..., 1]

    def l0_estimate(self) -> torch.Tensor:
        return self.keep_probs.sum()


def apply_concept_mask(
    z_sae: torch.Tensor,      # (N*H*W, nb_concepts)
    mask: torch.Tensor,        # (nb_concepts,)
) -> torch.Tensor:
    return z_sae * mask.unsqueeze(0)   # broadcast over tokens




def forward_pass(x, y, mask, backbone, sae):
    n = x.size(0)
    with torch.no_grad():
        z_raw  = extract_features(backbone, x)
        z_norm = sae.normalizer(z_raw)
        _, _, h, w = z_norm.shape
        z_flat = rearrange(z_norm, "n c h w -> (n h w) c")
        _, z_sae = sae.encode(z_flat)
        z_sae = z_sae.detach().clone()
        
    # --- grad flows from here through mask ---
    z_masked_flat       = apply_concept_mask(z_sae, mask)
    
    dictionary = sae.get_dictionary().detach()   # (nb_concepts, in_dim)
    z_masked_recon_flat = z_masked_flat @ dictionary        # grad flows through z_masked_flat only
    
    #z_masked_recon_flat = sae.decode(z_masked_flat)
    z_masked_recon      = rearrange(z_masked_recon_flat, "(n h w) c -> n c h w", n=n, h=h, w=w)
    z_masked_recon      = sae.normalizer.denormalize(z_masked_recon)
    logits              = backbone.classifier(pool(z_masked_recon).flatten(1))

    return F.cross_entropy(logits, y)

def train_concept_mask(backbone, sae, source_loader, target_loader, nb_concepts,
    n_steps=500, lr=1e-3,
    tau_start=2.0, tau_end=0.1,
):
    backbone.eval()
    sae.eval()

    for p in backbone.parameters(): p.requires_grad_(False)
    for p in sae.parameters():      p.requires_grad_(False)

    mask_module = GumbelConceptMask(nb_concepts).to(device)
    optimizer   = torch.optim.Adam(mask_module.parameters(), lr=lr)

    pbar = tqdm(zip(cycle(source_loader), cycle(target_loader)), total=n_steps, desc="Training concept mask")

    for step, ((x_src, y_src), (x_tgt, y_tgt)) in enumerate(pbar):
        if step >= n_steps:
            break

        x_src, y_src = x_src.to(device), y_src.to(device)
        x_tgt, y_tgt = x_tgt.to(device), y_tgt.to(device)

        tau = tau_start * (tau_end / tau_start) ** (step / max(n_steps - 1, 1))

        optimizer.zero_grad()
        l_l0  = mask_module.keep_probs.detach().sum()
        
        mask  = mask_module(tau=tau, hard=False)
        l_src = forward_pass(x_src, y_src, mask, backbone, sae)
        l_tgt = forward_pass(x_tgt, y_tgt, mask, backbone, sae)
        loss  = l_src - 3 * l_tgt +  1e-1 * l_l0
        loss.backward()
        optimizer.step()

        with torch.no_grad():
            n_zeroed = (~(mask_module.keep_probs > 0.5)).sum().item()
        pbar.set_postfix(loss=f"{loss.item():.3f}", src=f"{l_src.item():.3f}",
                         tgt=f"{l_tgt.item():.3f}", l10=f"{l_l0.item():.3f}", tau=f"{tau:.3f}",
                         zeroed=f"{int(n_zeroed)}/{nb_concepts}")

    with torch.no_grad():
        zeroed_concepts = (~(mask_module.keep_probs > 0.5)).nonzero(as_tuple=True)[0]

    print(f"\nInvariant concept indices: {zeroed_concepts.tolist()}")
    return mask_module, zeroed_concepts

In [ ]:
ckpt = 3300

# Do this once, right after loading your models
backbone = backbones[ckpt].to(device).eval()
sae      = SAEs[ckpt].eval()

for p in backbone.parameters(): p.requires_grad_(False)
for p in sae.parameters():      p.requires_grad_(False)

# verify
assert all(not p.requires_grad for p in backbone.parameters())
assert all(not p.requires_grad for p in sae.parameters())
print("All frozen ✓")

# NOW call train
source_loader, _ = Load_PACS(domains=["art_painting", "photo", "cartoon"])
target_loader, _ = Load_PACS(domains=["sketch"])

mask, zero_concepts = train_concept_mask(backbone, sae, source_loader, target_loader, 2048*8,
    n_steps=500, lr=1e-2,
    tau_start=2.0, tau_end=0.1,
)


In [ ]:
def masked_accuracy_by_domain(
    model,
    sae,
    domain_loaders: dict,          # {"art_painting": loader, "sketch": loader, ...}
    concepts_to_mask: torch.Tensor, # 1D tensor of concept indices to zero out
    num_classes: int = 7,
    nb_concepts: int = 16384,
    device: str = 'cuda',
) -> dict:

    model.eval()
    sae.eval()
    pool = SelectAdaptivePool2d(pool_type='avg', flatten=True)

    # build boolean mask once
    concept_mask = torch.zeros(nb_concepts, dtype=torch.bool, device=device)
    concept_mask[concepts_to_mask] = True

    results = {}

    for domain_name, dataloader in domain_loaders.items():

        correct_baseline = torch.zeros(num_classes, device=device)
        correct_masked   = torch.zeros(num_classes, device=device)
        total_per_class  = torch.zeros(num_classes, device=device)

        with torch.no_grad():
            for x, y in tqdm(dataloader, desc=f"{domain_name}"):
                x, y = x.to(device), y.to(device)
                n = x.size(0)

                z_raw  = extract_features(model, x)
                z_norm = sae.normalizer(z_raw)
                _, _, h, w = z_norm.shape
                z_flat   = rearrange(z_norm, "n c h w -> (n h w) c")
                _, z_sae = sae.encode(z_flat)

                # baseline
                z_recon      = sae.decode(z_sae)
                z_recon      = rearrange(z_recon, "(n h w) c -> n c h w", n=n, h=h, w=w)
                z_recon      = sae.normalizer.denormalize(z_recon)
                preds_base   = model.classifier(pool(z_recon)).argmax(dim=1)

                # masked
                z_sae_masked = z_sae.clone()
                z_sae_masked[:, concept_mask] = 0.0
                z_recon_m    = sae.decode(z_sae_masked)
                z_recon_m    = rearrange(z_recon_m, "(n h w) c -> n c h w", n=n, h=h, w=w)
                z_recon_m    = sae.normalizer.denormalize(z_recon_m)
                preds_masked = model.classifier(pool(z_recon_m)).argmax(dim=1)

                for cls in range(num_classes):
                    cls_mask = (y == cls)
                    correct_baseline[cls] += (preds_base[cls_mask]   == y[cls_mask]).sum()
                    correct_masked[cls]   += (preds_masked[cls_mask] == y[cls_mask]).sum()
                    total_per_class[cls]  += cls_mask.sum()

        per_class_base   = (correct_baseline / total_per_class.clamp(min=1)).cpu()
        per_class_masked = (correct_masked   / total_per_class.clamp(min=1)).cpu()

        results[domain_name] = {
            "per_class_baseline_accuracy": {c: per_class_base[c].item()   for c in range(num_classes)},
            "per_class_masked_accuracy":   {c: per_class_masked[c].item() for c in range(num_classes)},
            "overall_baseline_accuracy":   (correct_baseline.sum() / total_per_class.sum()).item(),
            "overall_masked_accuracy":     (correct_masked.sum()   / total_per_class.sum()).item(),
            "overall_delta":               ((correct_masked.sum() - correct_baseline.sum()) / total_per_class.sum()).item(),
        }

    results["concepts_masked"] = concepts_to_mask.tolist()
    results["num_concepts_masked"] = len(concepts_to_mask)
    return results


def print_masked_accuracy_table(results: dict, title: str = "MASKED CONCEPT ACCURACY"):

    domain_results = {k: v for k, v in results.items() 
                      if k not in ("concepts_masked", "num_concepts_masked")}

    n_masked = results.get("num_concepts_masked", "?")

    print(f"\n{'═'*62}")
    print(f"  {title}")
    print(f"  Concepts zeroed: {n_masked}")
    print(f"{'═'*62}")
    print(f"  {'Domain':<20} {'Baseline':>10} {'Masked':>10} {'Δ':>8}")
    print(f"  {'─'*54}")

    for domain, metrics in domain_results.items():
        b    = metrics["overall_baseline_accuracy"] * 100
        m    = metrics["overall_masked_accuracy"]   * 100
        d    = m - b
        sign = "+" if d >= 0 else ""
        print(f"  {domain.replace('_', ' ').title():<20} {b:>9.2f}% {m:>9.2f}% {sign}{d:>6.2f}%")

    print(f"  {'─'*54}")

    avg_b = sum(m["overall_baseline_accuracy"] for m in domain_results.values()) / len(domain_results) * 100
    avg_m = sum(m["overall_masked_accuracy"]   for m in domain_results.values()) / len(domain_results) * 100
    avg_d = avg_m - avg_b
    sign  = "+" if avg_d >= 0 else ""
    print(f"  {'Average':<20} {avg_b:>9.2f}% {avg_m:>9.2f}% {sign}{avg_d:>6.2f}%")
    print(f"{'═'*62}\n")

In [ ]:
domain_loaders = {
    "art_painting": Load_PACS(domains=["art_painting"])[0],
    "photo":        Load_PACS(domains=["photo"])[0],
    "cartoon":      Load_PACS(domains=["cartoon"])[0],
    "sketch":       Load_PACS(domains=["sketch"])[0],
}

results = masked_accuracy_by_domain(
    backbone, sae, domain_loaders,
    concepts_to_mask=zero_concepts,
    num_classes=7, nb_concepts=2048*8,
)

print_masked_accuracy_table(results, title="MASKING INVARIANT CONCEPTS")

## Per Class Drops

In [ ]:
class GumbelConceptMaskPerClass(nn.Module):
    def __init__(self, nb_concepts: int, num_classes: int, init_logit: float = 2.0):
        super().__init__()
        self.num_classes = num_classes
        # (num_classes, nb_concepts, 2)
        self.logits = nn.Parameter(
            torch.stack([
                torch.full((num_classes, nb_concepts), -init_logit),
                torch.full((num_classes, nb_concepts),  init_logit),
            ], dim=-1)
        )

    def forward(self, cls: int, tau: float = 1.0, hard: bool = False) -> torch.Tensor:
        # returns mask for a single class: (nb_concepts,)
        mask_soft = F.gumbel_softmax(self.logits[cls], tau=tau, hard=hard, dim=-1)
        return mask_soft[..., 1]

    def keep_probs_for_class(self, cls: int) -> torch.Tensor:
        return torch.softmax(self.logits[cls], dim=-1)[..., 1]

    @property
    def keep_probs_all(self) -> torch.Tensor:
        # (num_classes, nb_concepts)
        return torch.softmax(self.logits, dim=-1)[..., 1]

    def l0_estimate(self, cls: int) -> torch.Tensor:
        return self.keep_probs_for_class(cls).sum()

def forward_pass_cls(x, y, cls, mask, backbone, sae):
    # filter to only samples of this class
    cls_mask = (y == cls)
    if cls_mask.sum() == 0:
        return None

    x_cls = x[cls_mask]
    y_cls = y[cls_mask]
    n     = x_cls.size(0)

    with torch.no_grad():
        z_raw  = extract_features(backbone, x_cls)
        z_norm = sae.normalizer(z_raw)
        _, _, h, w = z_norm.shape
        z_flat = rearrange(z_norm, "n c h w -> (n h w) c")
        _, z_sae = sae.encode(z_flat)
        z_sae = z_sae.detach().clone()

    z_masked_flat       = apply_concept_mask(z_sae, mask)
    dictionary          = sae.get_dictionary().detach()
    z_masked_recon_flat = z_masked_flat @ dictionary
    z_masked_recon      = rearrange(z_masked_recon_flat, "(n h w) c -> n c h w", n=n, h=h, w=w)
    z_masked_recon      = sae.normalizer.denormalize(z_masked_recon)
    logits              = backbone.classifier(pool(z_masked_recon).flatten(1))

    return F.cross_entropy(logits, y_cls)


def train_concept_mask_per_class(backbone, sae, source_loader, target_loader,
    nb_concepts, num_classes=7,
    n_steps=500, lr=1e-3, lambda_l0=3e-1,
    tau_start=2.0, tau_end=0.1,
):
    backbone.eval()
    sae.eval()

    for p in backbone.parameters(): p.requires_grad_(False)
    for p in sae.parameters():      p.requires_grad_(False)

    mask_module = GumbelConceptMaskPerClass(nb_concepts, num_classes).to(device)
    optimizer   = torch.optim.Adam(mask_module.parameters(), lr=lr)

    pbar = tqdm(zip(cycle(source_loader), cycle(target_loader)),
                total=n_steps, desc="Training per-class concept mask")

    for step, ((x_src, y_src), (x_tgt, y_tgt)) in enumerate(pbar):
        if step >= n_steps:
            break

        x_src, y_src = x_src.to(device), y_src.to(device)
        x_tgt, y_tgt = x_tgt.to(device), y_tgt.to(device)

        tau = tau_start * (tau_end / tau_start) ** (step / max(n_steps - 1, 1))

        optimizer.zero_grad()

        total_loss = torch.tensor(0.0, device=device)
        classes_seen = 0

        for cls in range(num_classes):
            # skip if this class absent from both batches
            if (y_src == cls).sum() == 0 and (y_tgt == cls).sum() == 0:
                continue

            mask  = mask_module(cls, tau=tau, hard=False)
            l_l0  = mask_module.keep_probs_for_class(cls).detach().sum()

            l_src = forward_pass_cls(x_src, y_src, cls, mask, backbone, sae)
            l_tgt = forward_pass_cls(x_tgt, y_tgt, cls, mask, backbone, sae)

            if l_src is None or l_tgt is None:
                continue

            total_loss  = total_loss + l_src - l_tgt + lambda_l0 * l_l0
            classes_seen += 1

        if classes_seen == 0:
            continue

        total_loss = total_loss / classes_seen   # normalise by classes present
        total_loss.backward()
        optimizer.step()

        with torch.no_grad():
            kept_all   = mask_module.keep_probs_all > 0.5  # (num_classes, nb_concepts)
            avg_zeroed = (~kept_all).float().sum(dim=1).mean().item()

        pbar.set_postfix(loss=f"{total_loss.item():.3f}", tau=f"{tau:.3f}",
                         avg_zeroed=f"{avg_zeroed:.0f}/{nb_concepts}")

    # extract per-class zeroed concepts
    with torch.no_grad():
        zeroed_per_class = {
            cls: (~(mask_module.keep_probs_for_class(cls) > 0.5)).nonzero(as_tuple=True)[0]
            for cls in range(num_classes)
        }

    for cls, indices in zeroed_per_class.items():
        print(f"  class {cls}: {len(indices)} concepts zeroed")

    return mask_module, zeroed_per_class

In [ ]:
ckpt = 3300

# Do this once, right after loading your models
backbone = backbones[ckpt].to(device).eval()
sae      = SAEs[ckpt].eval()

for p in backbone.parameters(): p.requires_grad_(False)
for p in sae.parameters():      p.requires_grad_(False)

# verify
assert all(not p.requires_grad for p in backbone.parameters())
assert all(not p.requires_grad for p in sae.parameters())
print("All frozen ✓")

# NOW call train
source_loader, _ = Load_PACS(domains=["art_painting", "photo", "cartoon"])
target_loader, _ = Load_PACS(domains=["sketch"])

mask, zero_concepts = train_concept_mask_per_class(backbone, sae, source_loader, target_loader, 2048*8,
    n_steps=500, lr=1e-2,
    tau_start=2.0, tau_end=0.1,
)


In [ ]:
def masked_accuracy_by_domain_per_class(
    model,
    sae,
    domain_loaders: dict,
    zeroed_per_class: dict,        # {cls: tensor of concept indices to zero}
    num_classes: int = 7,
    nb_concepts: int = 16384,
    device: str = 'cuda',
) -> dict:

    model.eval()
    sae.eval()
    pool = SelectAdaptivePool2d(pool_type='avg', flatten=True)

    # build per-class boolean masks
    concept_masks = {}
    for cls, indices in zeroed_per_class.items():
        m = torch.zeros(nb_concepts, dtype=torch.bool, device=device)
        if len(indices) > 0:
            m[indices] = True
        concept_masks[cls] = m

    results = {}

    for domain_name, dataloader in domain_loaders.items():

        correct_baseline = torch.zeros(num_classes, device=device)
        correct_masked   = torch.zeros(num_classes, device=device)
        total_per_class  = torch.zeros(num_classes, device=device)

        with torch.no_grad():
            for x, y in tqdm(dataloader, desc=f"{domain_name}"):
                x, y = x.to(device), y.to(device)
                n = x.size(0)

                z_raw  = extract_features(model, x)
                z_norm = sae.normalizer(z_raw)
                _, _, h, w = z_norm.shape
                z_flat   = rearrange(z_norm, "n c h w -> (n h w) c")
                _, z_sae = sae.encode(z_flat)

                # baseline — no masking
                z_recon    = sae.decode(z_sae)
                z_recon    = rearrange(z_recon, "(n h w) c -> n c h w", n=n, h=h, w=w)
                z_recon    = sae.normalizer.denormalize(z_recon)
                preds_base = model.classifier(pool(z_recon)).argmax(dim=1)

                # masked — apply each sample's class-specific mask
                z_sae_masked = rearrange(z_sae.clone(), "(n h w) c -> n h w c", n=n, h=h, w=w)

                for cls in range(num_classes):
                    cls_idx = (y == cls)
                    if cls_idx.sum() == 0:
                        continue
                    # slice out, zero, put back
                    z_cls = z_sae_masked[cls_idx]                    # (n_cls, h, w, nb_concepts)
                    z_cls[..., concept_masks[cls]] = 0.0
                    z_sae_masked[cls_idx] = z_cls

                z_sae_masked = rearrange(z_sae_masked, "n h w c -> (n h w) c")

                z_recon_m    = sae.decode(z_sae_masked)
                z_recon_m    = rearrange(z_recon_m, "(n h w) c -> n c h w", n=n, h=h, w=w)
                z_recon_m    = sae.normalizer.denormalize(z_recon_m)
                preds_masked = model.classifier(pool(z_recon_m)).argmax(dim=1)

                for cls in range(num_classes):
                    cls_mask = (y == cls)
                    correct_baseline[cls] += (preds_base[cls_mask]   == y[cls_mask]).sum()
                    correct_masked[cls]   += (preds_masked[cls_mask] == y[cls_mask]).sum()
                    total_per_class[cls]  += cls_mask.sum()

        per_class_base   = (correct_baseline / total_per_class.clamp(min=1)).cpu()
        per_class_masked = (correct_masked   / total_per_class.clamp(min=1)).cpu()

        results[domain_name] = {
            "per_class_baseline_accuracy": {c: per_class_base[c].item()   for c in range(num_classes)},
            "per_class_masked_accuracy":   {c: per_class_masked[c].item() for c in range(num_classes)},
            "overall_baseline_accuracy":   (correct_baseline.sum() / total_per_class.sum()).item(),
            "overall_masked_accuracy":     (correct_masked.sum()   / total_per_class.sum()).item(),
            "overall_delta":               ((correct_masked.sum() - correct_baseline.sum()) / total_per_class.sum()).item(),
        }

    results["zeroed_per_class"] = {cls: indices.tolist() for cls, indices in zeroed_per_class.items()}
    results["num_zeroed_per_class"] = {cls: len(indices) for cls, indices in zeroed_per_class.items()}
    return results


def print_masked_accuracy_table_per_class(results: dict, class_names: dict = None,
                                          title: str = "PER-CLASS MASKED CONCEPT ACCURACY"):

    domain_results = {k: v for k, v in results.items()
                      if k not in ("zeroed_per_class", "num_zeroed_per_class")}
    num_zeroed     = results.get("num_zeroed_per_class", {})
    num_classes    = len(next(iter(domain_results.values()))["per_class_baseline_accuracy"])

    def cls_name(c):
        if class_names and c in class_names:
            return class_names[c]
        return f"class {c}"

    print(f"\n{'═'*72}")
    print(f"  {title}")
    print(f"{'═'*72}")

    # ── overall accuracy table ────────────────────────────────────────────
    print(f"\n  {'Domain':<20} {'Baseline':>10} {'Masked':>10} {'Δ':>8}")
    print(f"  {'─'*54}")
    for domain, metrics in domain_results.items():
        b    = metrics["overall_baseline_accuracy"] * 100
        m    = metrics["overall_masked_accuracy"]   * 100
        d    = m - b
        sign = "+" if d >= 0 else ""
        print(f"  {domain.replace('_',' ').title():<20} {b:>9.2f}% {m:>9.2f}% {sign}{d:>6.2f}%")
    print(f"  {'─'*54}")
    avg_b = sum(m["overall_baseline_accuracy"] for m in domain_results.values()) / len(domain_results) * 100
    avg_m = sum(m["overall_masked_accuracy"]   for m in domain_results.values()) / len(domain_results) * 100
    avg_d = avg_m - avg_b
    print(f"  {'Average':<20} {avg_b:>9.2f}% {avg_m:>9.2f}% {'+' if avg_d>=0 else ''}{avg_d:>6.2f}%")

    # ── per-class breakdown ───────────────────────────────────────────────
    print(f"\n{'═'*72}")
    print(f"  PER-CLASS BREAKDOWN")
    print(f"{'═'*72}")

    for cls in range(num_classes):
        n_zeroed = num_zeroed.get(cls, "?")
        print(f"\n  {cls_name(cls).upper()}  (concepts zeroed: {n_zeroed})")
        print(f"  {'─'*54}")
        print(f"  {'Domain':<20} {'Baseline':>10} {'Masked':>10} {'Δ':>8}")
        print(f"  {'─'*54}")
        for domain, metrics in domain_results.items():
            b    = metrics["per_class_baseline_accuracy"][cls] * 100
            m    = metrics["per_class_masked_accuracy"][cls]   * 100
            d    = m - b
            sign = "+" if d >= 0 else ""
            print(f"  {domain.replace('_',' ').title():<20} {b:>9.2f}% {m:>9.2f}% {sign}{d:>6.2f}%")
        print(f"  {'─'*54}")

    print(f"\n{'═'*72}\n")

In [ ]:
class_names = {0: "Dog", 1: "Elephant", 2: "Giraffe", 3: "Guitar", 
               4: "Horse", 5: "House", 6: "Person"}

results = masked_accuracy_by_domain_per_class(
    backbone, sae, domain_loaders, zero_concepts,
    num_classes=7, nb_concepts=2048*8,
)

print_masked_accuracy_table_per_class(results, class_names=class_names)

## Intervention

In [ ]:
import json
from typing import List, Tuple, Optional

def _in_any_interval(value: float, intervals: Optional[List[Tuple[float, float]]]) -> bool:
    """Return True if value falls within ANY of the given [min, max] intervals.
    If intervals is None or empty, the check is skipped (always passes)."""
    if not intervals:
        return True
    return any(lo <= value <= hi for lo, hi in intervals)

def train_backbone_suppress_negative_concepts(
    model,
    sae,
    dataloader,
    json_filepath: str,
    generalization_score_intervals: Optional[List[Tuple[float, float]]] = None,
    entropy_intervals:              Optional[List[Tuple[float, float]]] = None,
    discrimination_intervals:       Optional[List[Tuple[float, float]]] = None,
    mean_act_intervals:             Optional[List[Tuple[float, float]]] = None,
    num_classes: int = 7,
    nb_concepts: int = 7680,
    n_epochs: int = 5,
    lr: float = 1e-4,
    lambda_suppress: float = 1.0,
    device: str = 'cuda',
):
    # ------------------------------------------------------------------ #
    # 1. Load JSON and build per-class suppression masks                   #
    # ------------------------------------------------------------------ #
    with open(json_filepath, 'r') as f:
        data = json.load(f)

    concept_data = data.get("thresholded_concept_entropies", {})

    per_class_suppress = {
        cls: torch.zeros(nb_concepts, dtype=torch.bool, device=device)
        for cls in range(num_classes)
    }

    for class_id_str, concepts in concept_data.items():
        cls = int(class_id_str)
        for c in concepts:
            entropy        = c.get("entropy", 0.0)
            discrimination = c.get("discrimination_score", 0.0)
            gen_score      = entropy * discrimination

            if not _in_any_interval(gen_score,      generalization_score_intervals): continue
            if not _in_any_interval(entropy,         entropy_intervals):              continue
            if not _in_any_interval(discrimination,  discrimination_intervals):       continue
            if not _in_any_interval(sum(c.get("mean_acts", 0.0)), mean_act_intervals): continue

            per_class_suppress[cls][c["concept_index"]] = True

    print("Concepts to suppress per class:")
    for cls in range(num_classes):
        print(f"  class {cls}: {per_class_suppress[cls].sum().item()} concepts")

    # ------------------------------------------------------------------ #
    # 2. Freeze SAE, unfreeze entire model                                 #
    # ------------------------------------------------------------------ #
    sae.eval()
    for p in sae.parameters():
        p.requires_grad_(False)

    model.train()
    for p in model.parameters():
        p.requires_grad_(True)

    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    pool      = SelectAdaptivePool2d(pool_type='avg', flatten=True)

    # ------------------------------------------------------------------ #
    # 3. Training loop                                                     #
    # ------------------------------------------------------------------ #

    dictionary = sae.get_dictionary().detach()   # (nb_concepts, in_dim)

    for epoch in range(n_epochs):

        total_loss     = 0.0
        total_ce       = 0.0
        total_suppress = 0.0
        n_batches      = 0

        pbar = tqdm(dataloader, desc=f"Epoch {epoch+1}/{n_epochs}")

        for x, y in pbar:
            x, y = x.to(device), y.to(device)
            n = x.size(0)

            optimizer.zero_grad()

            # --- forward through backbone (grad enabled) ---
            if hasattr(model, 'featurizer'): # for models trained using domainbed
                if model.featurizer.__class__.__name__ == "DinoV2" :
                    activations = model.featurizer.network.forward_features(x.to(device))['x_norm_patchtokens']
                elif model.featurizer.__class__.__name__ == "ViT":
                    activations = model.featurizer.network.forward_features(x.to(device))[:, 1:, :]
                else:
                    activations = model.network[0](x.to(device))

            if hasattr(model, 'forward_features'): # for models directly from the overcomplete library
                activations = model.forward_features(x.to(device))



            z_norm = sae.normalizer(activations)
            _, _, h, w = z_norm.shape
            z_flat = rearrange(z_norm, "n c h w -> (n h w) c")

            # single encode pass with grad — flows back to backbone
            _, z_sae = sae.encode(z_flat)

            # --- CE loss ---
            
            logits  = model.classifier(pool(activations).flatten(1))
            l_ce    = F.cross_entropy(logits, y)

            # --- suppression loss ---
            z_sae_spatial = rearrange(z_sae, "(n h w) c -> n h w c", n=n, h=h, w=w)
            l_suppress    = torch.tensor(0.0, device=device)

            for cls in range(num_classes):
                cls_idx = (y == cls)
                if cls_idx.sum() == 0 or per_class_suppress[cls].sum() == 0:
                    continue

                z_cls_bad  = z_sae_spatial[cls_idx][..., per_class_suppress[cls]]
                l_suppress = l_suppress + z_cls_bad.abs().mean()

            loss = l_ce + lambda_suppress * l_suppress
            loss.backward()
            optimizer.step()

            total_loss     += loss.item()
            total_ce       += l_ce.item()
            total_suppress += l_suppress.item()
            n_batches      += 1

            pbar.set_postfix(
                loss=f"{loss.item():.4f}",
                ce=f"{l_ce.item():.4f}",
                suppress=f"{l_suppress.item():.4f}",
            )

        print(f"Epoch {epoch+1} | "
              f"loss={total_loss/n_batches:.4f}  "
              f"ce={total_ce/n_batches:.4f}  "
              f"suppress={total_suppress/n_batches:.4f}")

    return model

In [ ]:
ckpt = 3300

source_loader, _ = Load_PACS(domains=["art_painting", "photo", "cartoon"])

model = train_backbone_suppress_negative_concepts(
    model            = backbones[ckpt].to(device),
    sae              = SAEs[ckpt],
    dataloader       = source_loader,
    json_filepath    = "./invariances/uGEN_ERM_ResNet_T3_step3300.json",
    discrimination_intervals = [(-1.0, 0.0)],   # only suppress negative discrimination
    mean_act_intervals = [(20.0, 200000.0)],   # only suppress high mean activations
    num_classes      = 7,
    nb_concepts      = 2048 * 8,
    n_epochs         = 25,
    lr               = 1e-3,
    lambda_suppress  = 10.0,
    device           = device,
)

# then evaluate


In [ ]:
domain_loaders = {
    "art_painting": Load_PACS(domains=["art_painting"])[0],
    "photo":        Load_PACS(domains=["photo"])[0],
    "cartoon":      Load_PACS(domains=["cartoon"])[0],
    "sketch":       Load_PACS(domains=["sketch"])[0],
}

results = masked_accuracy_by_domain(
    model, SAEs[ckpt], domain_loaders,
    concepts_to_mask = torch.tensor([], dtype=torch.long),  # no masking at eval
    num_classes=7, nb_concepts=2048*8,
)
print_masked_accuracy_table(results, title="AFTER SUPPRESSION TRAINING")

step 3330

A: 0.9731051345

C: 0.9615384615  

P: 0.9850299401  

S: 0.8076433121    
